<a href="https://colab.research.google.com/github/lorenzo-stacchio/Deep-Learning-and-Computer-Vision-for-Business/blob/main/02-Pytorch%20and%20CV/02_detection/yolov12_retail_products/yolov12_retail_products.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv12 on the `retail_products` dataset

This notebook takes the retail product dataset shipped with the course
(`02-Pytorch and CV/datasets/retail_products`) and runs the full object detection
pipeline on it with **YOLOv12**: inference with the COCO-pretrained weights,
conversion of the annotations, fine-tuning, evaluation, and inference with the
fine-tuned model.

## What is YOLOv12?

Every YOLO release up to v11 is built almost entirely on convolutions. Attention
has better modelling capacity, but plain self-attention is quadratic in the number
of pixels and has poor memory access patterns, which is why it kept losing to CNNs
on the real-time speed/accuracy trade-off.

YOLOv12 ([Tian et al., 2025](https://arxiv.org/abs/2502.12524)) is the
*attention-centric* answer to that. The two ideas to remember:

| Component | What it does |
|---|---|
| **Area Attention (A2)** | splits the feature map into a few large horizontal/vertical regions and attends inside each one. Keeps a wide receptive field while cutting the cost of full self-attention. |
| **R-ELAN** | a residual variant of ELAN with scaled residual connections, which is what makes the larger attention-based models train stably. |

On top of that YOLOv12 drops positional encoding, enlarges the MLP ratio, and can
optionally use FlashAttention on recent NVIDIA GPUs (Turing, Ampere, Ada, Hopper).
**FlashAttention is not required** - Ultralytics falls back to a standard attention
implementation, it is only slower.

Checkpoints, from smallest to largest: `yolo12n.pt`, `yolo12s.pt`, `yolo12m.pt`,
`yolo12l.pt`, `yolo12x.pt`. Only *detection* weights are released; segmentation,
pose and classification variants exist as `.yaml` configs and must be trained from
scratch.

> **A note on model choice.** YOLOv12 is a community-driven release: Ultralytics
> flags it for possible training instability and higher memory usage, and points
> at YOLO11/YOLO26 for production workloads. We use it here because it is the
> clearest example of attention entering the real-time detection family - the
> pipeline below is identical for any other YOLO checkpoint, just change the
> model name.

## The dataset

380 images (512x512) of 6 Indonesian supermarket products, annotated in **Pascal VOC**
XML format:

| split | images | contents |
|---|---|---|
| `train` | 294 | 237 single-product shots + 57 "mix" shots with all 6 products |
| `test` | 86 | same composition, used here as the validation set |

Classes: `aqua` (water), `chitato` (crisps), `indomie` (instant noodles),
`pepsodent` (toothpaste), `shampoo`, `tissue`.

> **Pro tip:** on Colab go to `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> `GPU`.
> Fine-tuning on CPU works but takes roughly an hour instead of a few minutes.

## 0. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
# YOLOv12 needs a recent Ultralytics release; -U makes sure Colab's preinstalled
# version is upgraded.
!pip install -q -U ultralytics

from IPython import display
display.clear_output()

# Do not send usage analytics to Ultralytics.
!yolo settings sync=False

import ultralytics
ultralytics.checks()

In [ ]:
import os
import random
import re
import shutil
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

IN_COLAB = "google.colab" in str(get_ipython())
DEVICE = 0 if torch.cuda.is_available() else "cpu"
SEED = 0

random.seed(SEED)
np.random.seed(SEED)

print(f"Running in Colab: {IN_COLAB}")
print(f"Device: {DEVICE}")

## 1. Get the dataset

The notebook works both locally (inside the cloned course repo) and on Colab.
Locally it walks up to the repo root and uses the dataset in place; on Colab it
does a *sparse* checkout that pulls only the dataset folder, so we do not download
the videos and the large notebooks of the rest of the repo.

In [ ]:
REPO_URL = "https://github.com/lorenzo-stacchio/Deep-Learning-and-Computer-Vision-for-Business.git"
DATASET_SUBPATH = Path("02-Pytorch and CV/datasets/retail_products")


def find_local_dataset(start):
    """Walk up from `start` looking for the dataset inside the course repo."""
    for parent in [start, *start.parents]:
        candidate = parent / DATASET_SUBPATH
        if (candidate / "annotations").is_dir():
            return candidate
    return None


VOC_ROOT = find_local_dataset(Path.cwd().resolve())

if VOC_ROOT is None:
    # Colab (or any machine without the repo): sparse checkout of just the dataset.
    clone_dir = Path("course_repo")
    if not clone_dir.exists():
        !git clone --filter=blob:none --sparse --depth 1 {REPO_URL} {clone_dir}
        !git -C {clone_dir} sparse-checkout set "02-Pytorch and CV/datasets/retail_products"
    VOC_ROOT = (clone_dir / DATASET_SUBPATH).resolve()

assert (VOC_ROOT / "annotations" / "train").is_dir(), f"Dataset not found at {VOC_ROOT}"
print(f"Pascal VOC dataset: {VOC_ROOT}")

for split in ("train", "test"):
    n_img = len(list((VOC_ROOT / "images" / split).glob("*.jpg")))
    n_xml = len(list((VOC_ROOT / "annotations" / split).glob("*.xml")))
    print(f"  {split:<6} {n_img} images / {n_xml} annotations")

## 2. Explore the Pascal VOC annotations

Before converting anything, it pays to read one annotation file and see exactly
what we are dealing with.

In [ ]:
sample_xml = sorted((VOC_ROOT / "annotations" / "train").glob("*.xml"))[0]
print(f"--- {sample_xml.name} ---")
print(sample_xml.read_text())

Two things to notice, and they are exactly the kind of trap that silently ruins a
conversion script:

1. **`<filename>` does not match the file on disk.** The XML says
   `20210518_201521`, the file is called `aqua (1).xml`. Pairing images and
   annotations must be done on the *stem of the XML file*, never on the
   `<filename>` tag.
2. **`<path>` is the annotator's own Windows path.** Useless to us.

The boxes themselves are in absolute pixels as `(xmin, ymin, xmax, ymax)`, which
is the part we actually need.

In [ ]:
def parse_voc(xml_path):
    """Read a VOC XML file -> (width, height, [(class, xmin, ymin, xmax, ymax), ...])."""
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    width = int(float(size.find("width").text))
    height = int(float(size.find("height").text))

    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        box = obj.find("bndbox")
        objects.append((
            name,
            float(box.find("xmin").text),
            float(box.find("ymin").text),
            float(box.find("xmax").text),
            float(box.find("ymax").text),
        ))

    return width, height, objects


# Class distribution and objects-per-image, per split.
rows = []
for split in ("train", "test"):
    for xml_path in sorted((VOC_ROOT / "annotations" / split).glob("*.xml")):
        width, height, objects = parse_voc(xml_path)
        for name, *box in objects:
            rows.append({
                "split": split,
                "image": xml_path.stem,
                "class": name,
                "n_objects": len(objects),
                "width": width,
                "height": height,
                "box_area_ratio": ((box[2] - box[0]) * (box[3] - box[1])) / (width * height),
            })

ann = pd.DataFrame(rows)
print(f"{len(ann)} annotated objects over {ann['image'].nunique()} images")
print(f"image sizes: {sorted(set(zip(ann['width'], ann['height'])))}")

display(pd.crosstab(ann["class"], ann["split"], margins=True))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pd.crosstab(ann["class"], ann["split"]).plot.bar(ax=axes[0], rot=45)
axes[0].set_title("Objects per class")
axes[0].set_xlabel("")

(ann.drop_duplicates("image")["n_objects"]
    .value_counts().sort_index()
    .plot.bar(ax=axes[1], rot=0, color="tab:orange"))
axes[1].set_title("Objects per image")
axes[1].set_xlabel("number of objects")

axes[2].hist(ann["box_area_ratio"], bins=30, color="tab:green")
axes[2].set_title("Box area / image area")
axes[2].set_xlabel("ratio")

plt.tight_layout()
plt.show()

The dataset is well balanced across classes, and it is essentially bimodal: most
images contain a single product, while the 57 `mix_*` images contain all six at
once. Objects are large relative to the frame, so this is an easy detection
problem - a good thing for a teaching example, but keep in mind that real shelf
images are much harder.

In [ ]:
def draw_voc_sample(xml_paths, img_dir, n=6, seed=SEED):
    """Plot images with their Pascal VOC boxes drawn on top."""
    picks = random.Random(seed).sample(list(xml_paths), n)

    fig, axes = plt.subplots(2, n // 2, figsize=(4 * (n // 2), 8))
    for ax, xml_path in zip(axes.ravel(), picks):
        image_path = next(img_dir.glob(xml_path.stem + ".*"))
        _, _, objects = parse_voc(xml_path)

        ax.imshow(Image.open(image_path))
        for name, xmin, ymin, xmax, ymax in objects:
            ax.add_patch(patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor="lime", facecolor="none",
            ))
            ax.text(xmin, ymin - 4, name, color="black", fontsize=9,
                    bbox=dict(facecolor="lime", edgecolor="none", pad=1))

        ax.set_title(xml_path.stem, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


train_xmls = sorted((VOC_ROOT / "annotations" / "train").glob("*.xml"))
draw_voc_sample(train_xmls, VOC_ROOT / "images" / "train", n=6)

# The multi-product images are the interesting ones.
draw_voc_sample([p for p in train_xmls if p.stem.startswith("mix")],
                VOC_ROOT / "images" / "train", n=6, seed=1)

## 3. Convert Pascal VOC to the YOLO format

Ultralytics does not read Pascal VOC. It expects one `.txt` file per image, with
one line per object:

```
<class_id> <x_center> <y_center> <width> <height>
```

where all four coordinates are **normalised to [0, 1]**, and a folder layout of:

```
retail_products_yolo/
|-- train/
|   |-- images/aqua_1.jpg
|   +-- labels/aqua_1.txt
|-- val/
|   |-- images/
|   +-- labels/
+-- data.yaml
```

The source split `test` becomes our `val` split: with only 380 images we use the
held-out 86 images both to monitor training and to report the final metrics.
That is a simplification - see the exercises at the end.

> The same code lives in `voc_to_yolo.py` next to this notebook, so it can be run
> as a standalone script: `python voc_to_yolo.py`.

In [ ]:
# The index in this list is the class id written into the .txt label files.
CLASSES = ["aqua", "chitato", "indomie", "pepsodent", "shampoo", "tissue"]

# "aqua (1).jpg" -> "aqua_1.jpg": spaces and parentheses are legal but awkward
# in paths, CLI commands and logs.
_SAFE = re.compile(r"[^A-Za-z0-9._-]+")


def safe_stem(stem):
    return _SAFE.sub("_", stem).strip("_")


def voc_box_to_yolo(box, width, height):
    """(xmin, ymin, xmax, ymax) in pixels -> (xc, yc, w, h) normalised to [0, 1]."""
    xmin, ymin, xmax, ymax = box

    # A few annotations can overshoot the image border by a pixel or two.
    xmin, xmax = max(0.0, min(xmin, xmax)), min(float(width), max(xmin, xmax))
    ymin, ymax = max(0.0, min(ymin, ymax)), min(float(height), max(ymin, ymax))

    return (
        ((xmin + xmax) / 2) / width,
        ((ymin + ymax) / 2) / height,
        (xmax - xmin) / width,
        (ymax - ymin) / height,
    )


def convert_split(src, dst, src_split, dst_split):
    img_src, ann_src = src / "images" / src_split, src / "annotations" / src_split
    img_dst, lbl_dst = dst / dst_split / "images", dst / dst_split / "labels"
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    stats = Counter()
    for xml_path in sorted(ann_src.glob("*.xml")):
        # Pair on the file stem, not on the <filename> tag.
        image_path = next((p for p in img_src.glob(xml_path.stem + ".*")), None)
        if image_path is None:
            stats["missing_images"] += 1
            continue

        width, height, objects = parse_voc(xml_path)

        lines = []
        for name, *box in objects:
            if name not in CLASSES:
                stats["unknown_class:" + name] += 1
                continue
            xc, yc, w, h = voc_box_to_yolo(box, width, height)
            if w <= 0 or h <= 0:
                stats["degenerate_boxes"] += 1
                continue
            lines.append(f"{CLASSES.index(name)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            stats[name] += 1

        stem = safe_stem(xml_path.stem)
        shutil.copy2(image_path, img_dst / (stem + image_path.suffix.lower()))
        (lbl_dst / (stem + ".txt")).write_text("\n".join(lines) + "\n", encoding="utf-8")
        stats["images"] += 1

    return stats


def write_data_yaml(dst):
    names = "\n".join(f"  {i}: {c}" for i, c in enumerate(CLASSES))
    yaml_path = dst / "data.yaml"
    yaml_path.write_text(
        f"path: {dst.resolve().as_posix()}\n"
        "train: train/images\n"
        "val: val/images\n"
        "test: val/images\n"
        "\n"
        "names:\n"
        f"{names}\n",
        encoding="utf-8",
    )
    return yaml_path

In [ ]:
YOLO_ROOT = Path("retail_products_yolo").resolve()

if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for src_split, dst_split in (("train", "train"), ("test", "val")):
    stats = convert_split(VOC_ROOT, YOLO_ROOT, src_split, dst_split)
    print(f"[{dst_split}] {stats['images']} images")
    for cls in CLASSES:
        print(f"    {cls:<12} {stats[cls]} boxes")
    issues = {k: v for k, v in stats.items() if k not in CLASSES and k != "images"}
    if issues:
        print(f"    issues: {issues}")

DATA_YAML = write_data_yaml(YOLO_ROOT)
print(f"\n--- {DATA_YAML} ---")
print(DATA_YAML.read_text())

### Sanity check the conversion

Never trust a coordinate conversion you have not drawn on screen. Here we read
back the generated `.txt` files, map the normalised coordinates to pixels, and
compare the object counts with the source annotations.

In [ ]:
# 1. Counts must match the Pascal VOC side exactly.
converted = Counter()
for label_path in YOLO_ROOT.rglob("*.txt"):
    for line in label_path.read_text().split("\n"):
        if line.strip():
            converted[CLASSES[int(line.split()[0])]] += 1

check = pd.DataFrame({
    "pascal_voc": ann["class"].value_counts(),
    "yolo": pd.Series(converted),
})
check["match"] = check["pascal_voc"] == check["yolo"]
display(check)
assert check["match"].all(), "Object counts differ between VOC and YOLO!"

# 2. All normalised coordinates must live in [0, 1].
values = np.array([
    [float(v) for v in line.split()[1:]]
    for label_path in YOLO_ROOT.rglob("*.txt")
    for line in label_path.read_text().split("\n") if line.strip()
])
print(f"\ncoordinate range: [{values.min():.4f}, {values.max():.4f}]")
assert values.min() >= 0 and values.max() <= 1, "Coordinates outside [0, 1]!"
print("Conversion verified.")

In [ ]:
def draw_yolo_sample(split="train", n=6, seed=1):
    """Read back the converted labels and draw them, de-normalising to pixels."""
    label_paths = sorted((YOLO_ROOT / split / "labels").glob("*.txt"))
    picks = random.Random(seed).sample(label_paths, n)

    fig, axes = plt.subplots(2, n // 2, figsize=(4 * (n // 2), 8))
    for ax, label_path in zip(axes.ravel(), picks):
        image_path = next((YOLO_ROOT / split / "images").glob(label_path.stem + ".*"))
        image = Image.open(image_path)
        W, H = image.size
        ax.imshow(image)

        for line in label_path.read_text().split("\n"):
            if not line.strip():
                continue
            cls_id, xc, yc, w, h = line.split()
            xc, yc, w, h = float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle(
                (xc - w / 2, yc - h / 2), w, h,
                linewidth=2, edgecolor="cyan", facecolor="none",
            ))
            ax.text(xc - w / 2, yc - h / 2 - 4, CLASSES[int(cls_id)],
                    color="black", fontsize=9,
                    bbox=dict(facecolor="cyan", edgecolor="none", pad=1))

        ax.set_title(label_path.stem, fontsize=10)
        ax.axis("off")

    plt.suptitle("Boxes read back from the converted YOLO labels", y=1.0)
    plt.tight_layout()
    plt.show()


draw_yolo_sample("train", n=6)

## 4. Baseline: YOLOv12 pretrained on COCO

First let us see what the off-the-shelf model does on our images. COCO has 80
classes, and `bottle` is one of them, but `chitato` or `pepsodent` obviously are
not - this is the concrete motivation for fine-tuning.

In [ ]:
MODEL_NAME = "yolo12n.pt"  # try yolo12s.pt / yolo12m.pt for more accuracy

baseline = YOLO(MODEL_NAME)
baseline.info()
print(f"\nCOCO classes: {len(baseline.names)}")

In [ ]:
val_images = sorted((YOLO_ROOT / "val" / "images").glob("*.jpg"))
demo_images = [p for p in val_images if p.stem.startswith("mix")][:2] or val_images[:2]

results = baseline.predict(source=[str(p) for p in demo_images], conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 7))
for ax, result in zip(np.atleast_1d(axes), results):
    ax.imshow(result.plot()[..., ::-1])  # plot() returns BGR
    detected = [baseline.names[int(c)] for c in result.boxes.cls]
    ax.set_title(f"COCO YOLOv12: {dict(Counter(detected)) or 'nothing detected'}", fontsize=10)
    ax.axis("off")

plt.suptitle("Pretrained model, before fine-tuning")
plt.tight_layout()
plt.show()

As expected: the model localises objects reasonably well (it was trained on
millions of images) but labels them with whatever COCO class is closest -
`bottle`, `book`, `cup`. The *what* is wrong, only the *where* is usable.
Fine-tuning re-teaches the head our six product classes.

## 5. Fine-tune YOLOv12

294 training images is a small dataset, so the settings below are chosen
accordingly:

* **start from COCO weights** rather than a `.yaml` config - training YOLOv12 from
  scratch on 294 images would not converge to anything useful;
* **`epochs=100` with `patience=25`**, since a small dataset needs many passes but
  will overfit if left running;
* **augmentation matters more than usual**. `mosaic` (stitching 4 images together)
  is what lets a dataset of mostly single-object images learn multi-object scenes;
  `close_mosaic=10` turns it off for the last 10 epochs so the model finishes on
  realistic images;
* **`batch=16, imgsz=640`** fits a Colab T4. Attention layers are memory-hungry, so
  on a smaller GPU this can run out of VRAM — Ultralytics detects that and retries
  with a halved batch, but you can also set `batch=-1` to have it sized
  automatically up front.

In [ ]:
training_args = {
    # Data and model
    "data": str(DATA_YAML),
    "epochs": 100,
    "patience": 25,             # early stopping if val mAP stops improving
    "batch": 16,                # -1 for auto-batch based on available VRAM
    "imgsz": 640,
    "device": DEVICE,
    "seed": SEED,
    # Windows spawns dataloader workers instead of forking them, which breaks
    # inside a notebook; 0 keeps loading in the main process. Colab is Linux.
    "workers": 0 if os.name == "nt" else 8,

    # Optimisation ("auto" picks AdamW with a tuned lr for small datasets)
    "optimizer": "auto",
    "lr0": 0.01,                # initial learning rate
    "lrf": 0.01,                # final lr = lr0 * lrf
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "cos_lr": True,             # cosine schedule, gentler on small datasets

    # Augmentation
    "hsv_h": 0.015,             # hue jitter: keep low, colour identifies the product
    "hsv_s": 0.7,               # saturation
    "hsv_v": 0.4,               # brightness: shops have very different lighting
    "degrees": 10.0,            # rotation
    "translate": 0.1,
    "scale": 0.5,               # zoom in/out
    "fliplr": 0.5,              # horizontal flip
    "flipud": 0.0,              # products are never upside down
    "mosaic": 1.0,              # key augmentation for multi-object generalisation
    "close_mosaic": 10,         # disable mosaic for the last 10 epochs
    "erasing": 0.4,             # random erasing, simulates occlusion on a shelf

    # Bookkeeping
    "project": "runs_retail",
    "name": "yolo12n_retail",
    "exist_ok": True,
    "plots": True,
    "val": True,
}

model = YOLO(MODEL_NAME)
n_train = len(list((YOLO_ROOT / "train" / "images").glob("*")))
print(f"Fine-tuning {MODEL_NAME} on {n_train} images")

In [ ]:
train_results = model.train(**training_args)

# Ultralytics resolves a relative `project` against its own runs directory, so
# do not rebuild the path by hand: ask the trainer where it actually saved.
RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print(f"\nRun directory: {RUN_DIR}")
print(f"Best weights:  {BEST_WEIGHTS} (exists: {BEST_WEIGHTS.exists()})")

## 6. Evaluate

The headline numbers for detection are:

* **mAP50** - mean average precision at IoU 0.5. "Did we find the object and name
  it right?" Forgiving about box tightness.
* **mAP50-95** - averaged over IoU thresholds 0.5 to 0.95. The strict metric, and
  the one to quote: it punishes loose boxes.
* **precision / recall** - false alarms vs missed products. For a retail
  checkout use case recall usually matters more: a missed item is unbilled stock.

In [ ]:
best = YOLO(BEST_WEIGHTS)
metrics = best.val(data=str(DATA_YAML), device=DEVICE, workers=training_args["workers"],
                   project=training_args["project"], name="val_best", exist_ok=True)

print(f"mAP50     {metrics.box.map50:.4f}")
print(f"mAP50-95  {metrics.box.map:.4f}")
print(f"precision {metrics.box.mp:.4f}")
print(f"recall    {metrics.box.mr:.4f}")

per_class = pd.DataFrame({
    "class": [best.names[int(c)] for c in metrics.box.ap_class_index],
    "precision": metrics.box.p,
    "recall": metrics.box.r,
    "mAP50": metrics.box.ap50,
    "mAP50-95": metrics.box.maps[metrics.box.ap_class_index],
}).set_index("class").sort_values("mAP50-95")

display(per_class.style.format("{:.4f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display as ipy_display

# Ultralytics renamed the metric curves at some point (PR_curve.png ->
# BoxPR_curve.png), so accept either spelling.
for title, candidates in [
    ("Training curves", ["results.png"]),
    ("Label distribution", ["labels.jpg"]),
    ("Confusion matrix (normalised)", ["confusion_matrix_normalized.png"]),
    ("Precision-Recall curve", ["BoxPR_curve.png", "PR_curve.png"]),
    ("F1 vs confidence", ["BoxF1_curve.png", "F1_curve.png"]),
    ("Validation batch: ground truth", ["val_batch0_labels.jpg"]),
    ("Validation batch: predictions", ["val_batch0_pred.jpg"]),
]:
    path = next((RUN_DIR / c for c in candidates if (RUN_DIR / c).exists()), None)
    if path is None:
        print(f"\n=== {title}: not found ({', '.join(candidates)}) ===")
        continue
    print(f"\n=== {title} ===")
    ipy_display(IPyImage(filename=str(path), width=850))

Reading the curves: `box_loss` is localisation quality, `cls_loss` is
classification, `dfl_loss` is the distribution focal loss on box edges. If the
validation losses turn upwards while the training ones keep falling, the model is
overfitting - with 294 images that is the failure mode to watch for, and the
answer is stronger augmentation or a smaller model, not more epochs.

## 7. Inference with the fine-tuned model

In [ ]:
demo = random.Random(7).sample(val_images, 6)
results = best.predict(source=[str(p) for p in demo], conf=0.35, iou=0.5, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, result in zip(axes.ravel(), results):
    ax.imshow(result.plot()[..., ::-1])
    ax.set_title(Path(result.path).stem, fontsize=10)
    ax.axis("off")

plt.suptitle("Fine-tuned YOLOv12 on the validation split")
plt.tight_layout()
plt.show()

### Predictions as a table

The `Results` object is convenient for plotting, but for any downstream business
logic (stock counting, basket totals, shelf audits) you want a dataframe. This is
the same shape of output used in the tracking notebook of module `03_tracking`.

In [ ]:
def predictions_to_frame(results, names):
    """Flatten a list of Ultralytics Results into one row per detected object."""
    rows = []
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            rows.append({
                "image": Path(result.path).stem,
                "class": names[int(box.cls)],
                "confidence": float(box.conf),
                "x1": round(x1), "y1": round(y1), "x2": round(x2), "y2": round(y2),
                "area": round((x2 - x1) * (y2 - y1)),
            })
    # Name the columns explicitly so the frame is still well-formed when the
    # model detects nothing - otherwise the groupby below fails with a KeyError.
    return pd.DataFrame(rows, columns=["image", "class", "confidence",
                                       "x1", "y1", "x2", "y2", "area"])


# `stream=True` yields one Result at a time instead of building the whole list in
# memory: pass a folder (or a video) this way and the run stays flat in VRAM,
# however many images it contains.
detections = predictions_to_frame(
    best.predict(source=str(YOLO_ROOT / "val" / "images"), conf=0.35,
                 stream=True, verbose=False),
    best.names,
)

print(f"{len(detections)} objects detected over {len(val_images)} validation images\n")
display(detections.head(10))

# A basic business read: what is on the shelf, and how confident are we?
summary = (detections.groupby("class")
           .agg(detections=("class", "size"), mean_confidence=("confidence", "mean"))
           .sort_values("detections", ascending=False))
display(summary.style.format({"mean_confidence": "{:.3f}"}))

### Inference on your own image

Point `source` at anything: a local file, a URL, a folder, a video, or a webcam
index. The API is identical.

In [ ]:
# Example: uncomment and set your own path or URL.
# custom = best.predict(source="https://example.com/shelf.jpg", conf=0.35, save=True)
# plt.figure(figsize=(10, 8))
# plt.imshow(custom[0].plot()[..., ::-1])
# plt.axis("off")
# plt.show()

## 8. Export the model

`best.pt` is a PyTorch checkpoint and needs PyTorch to run. For deployment
(an in-store camera, an edge device, a checkout terminal) you usually export to a
runtime-specific format. ONNX is the portable default; `engine` (TensorRT),
`coreml`, `tflite` and `openvino` are also available.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=640, simplify=True)
print(f"Exported to: {onnx_path}")

# Reloading works through the same API, whatever the format.
deployed = YOLO(onnx_path)
check_result = deployed.predict(source=str(val_images[0]), conf=0.35, verbose=False)
print(f"ONNX detections: {[deployed.names[int(c)] for c in check_result[0].boxes.cls]}")

In [ ]:
# On Colab, download the weights before the runtime is recycled.
if IN_COLAB:
    from google.colab import files
    files.download(str(BEST_WEIGHTS))

## 9. Exercises

1. **Proper three-way split.** We used the 86 test images as the validation set,
   which means the reported mAP is measured on data also used for early stopping.
   Carve a real validation set out of the 294 training images and keep the 86 as
   an untouched test set. How much does the reported mAP50-95 drop?

2. **Model size.** Re-run with `yolo12s.pt` and `yolo12m.pt`. Plot mAP50-95
   against inference time per image (`result.speed`). Where is the knee of the
   curve for this dataset?

3. **How much data do you actually need?** Retrain on 25%, 50% and 75% of the
   training images. Plot mAP against dataset size - on a problem this easy the
   curve flattens surprisingly early, which is a useful number to have when
   costing an annotation campaign.

4. **Augmentation ablation.** Set `mosaic=0.0` and retrain. The `mix_*` images are
   the only multi-object examples in the dataset; measure how much mosaic is
   carrying the model's ability to handle crowded scenes.

5. **Compare against the rest of the course.** Run the same dataset through the
   YOLO11 pipeline in `02_detection/yolo_custom_dataset/` and through MMDetection
   in `02_detection/mmdetection/`. Same data, three frameworks.

6. **Close the loop with tracking.** Take `best.pt` into `03_tracking/` and run it
   with `model.track()` on a video of a shelf or a checkout counter, then count
   unique products the way the tracking notebook counts people.